# វិទ្យាសាស្ត្រទិន្នន័យនៅក្នុងពពក៖ វិធី "Azure ML SDK"

## ការប្រកាសមុខ

ក្នុងសៀវភៅកំណត់ត្រានេះ យើងនឹងរៀនពីរបៀបប្រើប្រាស់ Azure ML SDK ដើម្បីហ្វឹកហ្វឺន បញ្ចូន និងប្រើម៉ូដែលតាមរយៈ Azure ML។

លក្ខខណ្ឌមុន៖  
1. អ្នកបានបង្កើតកន្លែងធ្វើការ Azure ML ។  
2. អ្នកបានផ្ទុក [ទិន្នន័យជំងឺបេះដូង](https://www.kaggle.com/andrewmvd/heart-failure-clinical-data) ទៅក្នុង Azure ML។  
3. អ្នកបានផ្ទុកសៀវភៅកំណត់ត្រានេះចូលក្នុង Azure ML Studio ។  

ជំហានបន្ទាប់មាន៖

1. បង្កើតការធ្វើតេស្តក្នុង Workspace មានស្រាប់។  
2. បង្កើតក្រុម Compute ។  
3. ផ្ទុកទិន្នន័យ។  
4. តំឡើង AutoML ដោយប្រើ AutoMLConfig។  
5. ប្រតិបត្តិការតេស្ត AutoML។  
6. ស្វែងរកលទ្ធផលនិងទទួលបានម៉ូដែលល្អបំផុត។  
7. ធ្វើការចុះបញ្ជីម៉ូដែលល្អបំផុត។  
8. បញ្ចូនម៉ូដែលល្អបំផុត។  
9. ប្រើប្រាស់ចំណុចចេញ។  

## ការនាំចេញជាក់លាក់សម្រាប់ Azure Machine Learning SDK


In [ ]:
from azureml.core import Workspace, Experiment
from azureml.core.compute import AmlCompute
from azureml.train.automl import AutoMLConfig
from azureml.widgets import RunDetails
from azureml.core.model import InferenceConfig, Model
from azureml.core.webservice import AciWebservice

## Initialize Workspace
ចាប់ផ្ដើមអក្សរការងារពីការកំណត់ដែលបានរក្សាទុក។ សូមធានាថា​ឯកសារ​កំណត់​មាន​នៅ​ក្នុង .\config.json


In [ ]:
ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep = '\n')

## បង្កើតការធ្វើតេស្ត Azure ML

យើងចាប់ផ្តើមបង្កើតការធ្វើតេស្តដែលមានឈ្មោះ 'aml-experiment' ក្នុងកន្លែងធ្វើការដែលយើងទើបតែចាប់ផ្តើម។


In [ ]:
experiment_name = 'aml-experiment'
experiment = Experiment(ws, experiment_name)
experiment

## បង្កើតក្រុមកុំព្យូទ័រ
អ្នកនឹងត្រូវបង្កើត [គោលដៅកុំព្យូទ័រ](https://docs.microsoft.com/azure/machine-learning/concept-azure-machine-learning-architecture#compute-target) សម្រាប់ការប្រតិបត្តិ AutoML របស់អ្នក។


In [ ]:
aml_name = "heart-f-cluster"
try:
    aml_compute = AmlCompute(ws, aml_name)
    print('Found existing AML compute context.')
except:
    print('Creating new AML compute context.')
    aml_config = AmlCompute.provisioning_configuration(vm_size = "Standard_D2_v2", min_nodes=1, max_nodes=3)
    aml_compute = AmlCompute.create(ws, name = aml_name, provisioning_configuration = aml_config)
    aml_compute.wait_for_completion(show_output = True)

cts = ws.compute_targets
compute_target = cts[aml_name]

## ទិន្នន័យ
ប្រាកដថាអ្នកបានផ្ទុកឡើងឯកសារទិន្នន័យទៅ Azure ML ហើយកូនសោមានឈ្មោះដូចគ្នានឹងឯកសារទិន្នន័យ។


In [ ]:
key = 'heart-failure-records'
dataset = ws.datasets[key]
df = dataset.to_pandas_dataframe()
df.describe()

## ការកំណត់ AutoML


In [ ]:
automl_settings = {
    "experiment_timeout_minutes": 20,
    "max_concurrent_iterations": 3,
    "primary_metric" : 'AUC_weighted'
}

automl_config = AutoMLConfig(compute_target=compute_target,
                             task = "classification",
                             training_data=dataset,
                             label_column_name="DEATH_EVENT",
                             enable_early_stopping= True,
                             featurization= 'auto',
                             debug_log = "automl_errors.log",
                             **automl_settings
                            )

## រត់ AutoML


In [ ]:
remote_run = experiment.submit(automl_config)

In [ ]:
RunDetails(remote_run).show()

## រក្សាទុកម៉ូដែលល្អបំផុត


In [ ]:
best_run, fitted_model = remote_run.get_output()

In [ ]:
best_run.get_properties()

In [ ]:
model_name = best_run.properties['model_name']
script_file_name = 'inference/score.py'
best_run.download_file('outputs/scoring_file_v_1_0_0.py', 'inference/score.py')
description = "aml heart failure project sdk"
model = best_run.register_model(model_name = model_name,
                                description = description,
                                tags = None)

## ចែកចាយម៉ូដែលល្អបំផុត

រត់កូដខាងក្រោមដើម្បីចែកចាយម៉ូដែលល្អបំផុត។ អ្នកអាចមើលស្ថានភាពនៃការចែកចាយនៅក្នុងទំព័រផ្ទៃតាបន្ទាត់ Azure ML ។ ជំហ៊ាននេះអាចចំណាយពេលប៉ុន្មាននាទី។


In [ ]:
inference_config = InferenceConfig(entry_script=script_file_name, environment=best_run.get_environment())

aciconfig = AciWebservice.deploy_configuration(cpu_cores = 1,
                                               memory_gb = 1,
                                               tags = {'type': "automl-heart-failure-prediction"},
                                               description = 'Sample service for AutoML Heart Failure Prediction')

aci_service_name = 'automl-hf-sdk'
aci_service = Model.deploy(ws, aci_service_name, [model], inference_config, aciconfig)
aci_service.wait_for_deployment(True)
print(aci_service.state)

## បរិច្ចាគចំណុចបញ្ចប់
អ្នកអាចបន្ថែមបញ្ចូលទៅលើគំរូបញ្ចូលខាងក្រោម។


In [ ]:
data = {
    "data":
    [
        {
            'age': "60",
            'anaemia': "false",
            'creatinine_phosphokinase': "500",
            'diabetes': "false",
            'ejection_fraction': "38",
            'high_blood_pressure': "false",
            'platelets': "260000",
            'serum_creatinine': "1.40",
            'serum_sodium': "137",
            'sex': "false",
            'smoking': "false",
            'time': "130",
        },
    ],
}

test_sample = str.encode(json.dumps(data))

In [ ]:
response = aci_service.run(input_data=test_sample)
response

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការផ្តល់អត្ថបទដោះស្រាយ**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះបីយើងខិតខំសំរាប់ភាពត្រឹមត្រូវក៏ដោយ សូមយល់ឲ្យបានថាការបកប្រែដោយស្វ័យប្រវត្តិនេះអាចមានកំហុស ឬការមិនត្រឹមត្រូវខ្លះ។ ឯកសារដើមនៅក្នុងភាសាមាតុភូមិគួរត្រូវបានគេសង្កេតជា ប្រភពមានសិទ្ធិ។ សម្រាប់ព័ត៌មានសំខាន់ណាស់ ការបកប្រែដោយអ្នកជំនាញដែលមានបទពិសោធន៍ត្រូវបានផ្តល់អនុសាសន៍។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែខុសពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
